# Agri Vision: EfficientDet optimized FP32 continuation - 71gmail

Use rehmanfx71@gmail.com, T4 GPU, then Run all. Keep 4gmail stopped. Five screening trials are complete; continue trial 6 from its last committed round (round 1 if none saved). This is the production training notebook, not the benchmark.

Only the three benchmarked FP32 overhead reductions are enabled. Original data, settings, precision, validation, checkpoint hashes and FedAvg remain unchanged. A separate execution profile fingerprints the extension and benchmark evidence; every new round records its version. All remaining screening, finalist confirmation and final training stages continue automatically. Earlier rounds retain original execution provenance.


In [ ]:
from pathlib import Path
DRIVE_ROOT = Path("/content/drive/MyDrive/Agri Vision")
WORKSPACE = Path("/content/agrivision_grid_v1")
OUTPUT = DRIVE_ROOT / "grid_search" / "grid_v1_20261017"
MODELS = ['efficientdet_d0']
WORKERS = 2
EXPECTED_DATASET_SHA256 = "7a2920e4a4e0405ee157e9b40ac6306b54d6933b20a306b44a4b7c6871f20868"
EXPECTED_CODE_SHA256 = "5453a56a97c2aca78224cec346c259e5f8a0a5eda2a93eb17b3677f06689c049"


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import hashlib, json, shutil, zipfile
assert DRIVE_ROOT.is_dir(), "Open this notebook as rehmanfx71@gmail.com; the copied Agri Vision folder must be in My Drive"
assert (DRIVE_ROOT / "71gmail_resume.json").is_file(), "Wrong Agri Vision folder: account migration marker missing"

def sha(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

WORKSPACE.mkdir(parents=True, exist_ok=True)
dataset_zip = WORKSPACE.parent / "non_chili_v1_colab_dataset.zip"
source_zip = DRIVE_ROOT / dataset_zip.name
if source_zip.is_file():
    if not dataset_zip.is_file() or sha(dataset_zip) != EXPECTED_DATASET_SHA256:
        shutil.copyfile(source_zip, dataset_zip)
else:
    if not dataset_zip.is_file() or sha(dataset_zip) != EXPECTED_DATASET_SHA256:
        parts_dir = DRIVE_ROOT / "non_chili_v1_upload_parts"
        manifest = json.loads((parts_dir / "parts.json").read_text())
        assert manifest["joined_sha256"] == EXPECTED_DATASET_SHA256, "Wrong dataset parts manifest"
        specs = manifest["parts"]
        assert specs and [p["name"] for p in specs] == [dataset_zip.name + f".part{i:02d}" for i in range(1, len(specs)+1)]
        partial = dataset_zip.with_suffix(".partial")
        with partial.open("wb") as output:
            for index, spec in enumerate(specs, 1):
                path = parts_dir / spec["name"]
                assert path.is_file() and path.stat().st_size == spec["bytes"], f"Missing/damaged part {index}"
                with path.open("rb") as stream:
                    shutil.copyfileobj(stream, output, 16*1024*1024)
                print(f"Dataset part {index}/{len(specs)} copied", flush=True)
        assert sha(partial) == EXPECTED_DATASET_SHA256, "Dataset parts damaged"
        partial.replace(dataset_zip)
assert sha(dataset_zip) == EXPECTED_DATASET_SHA256
dataset_marker = WORKSPACE / ".dataset_verified"
if not dataset_marker.exists() or dataset_marker.read_text() != EXPECTED_DATASET_SHA256:
    with zipfile.ZipFile(dataset_zip) as bundle:
        for name in bundle.namelist():
            assert (WORKSPACE/name).resolve().is_relative_to(WORKSPACE.resolve())
        bundle.extractall(WORKSPACE)
    dataset_marker.write_text(EXPECTED_DATASET_SHA256)
code_zip = WORKSPACE.parent / "agrivision_grid_search_v1_code.zip"
shutil.copyfile(DRIVE_ROOT/code_zip.name, code_zip)
assert sha(code_zip) == EXPECTED_CODE_SHA256, "Wrong grid-search code bundle"
with zipfile.ZipFile(code_zip) as bundle:
    for name in bundle.namelist():
        assert (WORKSPACE/name).resolve().is_relative_to(WORKSPACE.resolve())
    bundle.extractall(WORKSPACE)
print("Verified local dataset and isolated grid-search code.")


In [ ]:
import os, subprocess, sys
os.chdir(WORKSPACE)
sys.path.insert(0, str(WORKSPACE))
import torch, torchvision
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > T4 GPU"
assert "T4" in torch.cuda.get_device_name(0), "Use T4 for the comparison"
torch_before, vision_before = torch.__version__, torchvision.__version__
print("GPU:", torch.cuda.get_device_name(0), "torch:", torch_before, "torchvision:", vision_before)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-r", "requirements-grid.txt"], check=True)
# Preserve the existing CUDA packages while installing supporting libraries.
subprocess.run([sys.executable, "-m", "pip", "install", "huggingface_hub", "safetensors", "packaging", "tqdm", "omegaconf==2.4.0",
                "matplotlib", "opencv-python", "requests", "scipy", "psutil", "polars>=0.20", "ultralytics-thop>=2.0.18"], check=True)
assert torch.__version__ == torch_before and torchvision.__version__ == vision_before
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_grid_search.py", "-v"], check=True)


In [ ]:
# Verify the copied search before allowing any training.
model_folder = OUTPUT / "efficientdet_d0"
assert model_folder.is_dir(), "Copied search is missing; do not start a fresh grid"
initial = model_folder / "initial.pt"
assert sha(initial) == "dd0a92a8261e15f8776deba607c6ef0b9bab4389f3705a656c4a8050b45c6fce", "Common initialization changed"
from importlib.metadata import version
expected = json.loads((model_folder / "trials/trial_01/settings.json").read_text())
actual = {"gpu": torch.cuda.get_device_name(0), "torch": str(torch.__version__),
          "packages": {name: version(name) for name in expected["hardware"]["packages"]}}
assert actual == expected["hardware"], f"Resume environment differs: {actual} versus {expected['hardware']}. Keep the saved settings unchanged."
for number in range(1, 6):
    trial = model_folder / "trials" / f"trial_{number:02d}"
    latest = json.loads((trial / "latest.json").read_text())
    assert latest["round"] >= 5, f"Trial {number} is incomplete in this copy"
for trial in sorted((model_folder / "trials").glob("trial_*")):
    settings = json.loads((trial / "settings.json").read_text())
    assert settings["initial_checkpoint"] == str(initial), "Saved initialization path differs"
    assert settings["initial_checkpoint_sha256"] == sha(initial)
    latest_path = trial / "latest.json"
    if latest_path.exists():
        latest = json.loads(latest_path.read_text())
        checkpoint = (trial / latest["checkpoint"]).resolve()
        assert checkpoint.is_relative_to(trial.resolve())
        assert sha(checkpoint) == latest["checkpoint_sha256"], f"Damaged checkpoint: {trial.name}"
        assert all((trial / "results/metrics" / f"round_{r:03d}.json").is_file() for r in range(1, latest["round"]+1))
        print(trial.name, "saved round", latest["round"], flush=True)
    else:
        print(trial.name, "no committed rounds; starts at round 1", flush=True)
print("Copied search and exact resume environment verified.", flush=True)


## Automatic search, confirmation and final training
Each model's `preflight.json` records batch/resolution compatibility. `screening_results.csv` ranks all sixteen five-round trials, and `finalist_results.csv` ranks the two ten-round finalists. `selected_configuration.json` identifies the winner. The runner then trains that configuration from scratch through round 30 under `final_training/`, retaining validation-best and latest weights.


In [ ]:
extension = DRIVE_ROOT / "efficientdet_fp32_continuation_v1.zip"
assert sha(extension) == "35e010acf0d7bfebd3ff585d7cc33bdabd9bc009371d9e01b2fcc4f27ca5820e", "Wrong optimized continuation bundle"
with zipfile.ZipFile(extension) as archive:
    for name in archive.namelist():
        assert (WORKSPACE / name).resolve().is_relative_to(WORKSPACE.resolve())
        assert name.startswith(("continuation/", "benchmarks/")), "Extension must not replace original training modules"
    archive.extractall(WORKSPACE)
print("Verified optimized FP32 extension; continuing existing EfficientDet search.", flush=True)
command = [sys.executable, "-u", "-m", "continuation.search", "--root", str(WORKSPACE),
           "--output", str(OUTPUT), "--models", *MODELS, "--workers", str(WORKERS)]
env = dict(os.environ, PYTHONUNBUFFERED="1")
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, bufsize=1, env=env) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    returncode = process.wait()
if returncode:
    raise subprocess.CalledProcessError(returncode, command)
print("Completed models:", MODELS)
print("Results:", OUTPUT)


## Resume and result locations
If the session stops, reopen this same notebook on the same account, select T4 and Run all. No changes to settings or model order are needed. The current phase is recorded in `grid_search/grid_v1_20261017/search_status.json`; each trial has its own status and committed `latest.json`. The second SSD model starts automatically after the first completes search and final training.

Trial validation CSVs and per-class round JSONs are retained. Screening stores latest-only weights to conserve Drive space; final training retains best and latest. GPU preflight weights are discarded. No held-out testing is performed. The five-/ten-round selection is budgeted screening, not proof of the best possible thirty-round configuration.
